conda env: neurolens

In [1]:
import hashlib
import boto3

session = boto3.Session(
    profile_name="hcp",
    region_name="us-east-1",
)

credentials = session.get_credentials()

if credentials is None:
    raise RuntimeError("No credentials resolved for profile 'hcp'.")

frozen = credentials.get_frozen_credentials()

print("Session profile:", session.profile_name)
print("Credential method:", credentials.method)
print("Access-key prefix:", frozen.access_key[:4])
print("Access-key length:", len(frozen.access_key))
print(
    "Access-key fingerprint:",
    hashlib.sha256(frozen.access_key.encode()).hexdigest()[:12],
)
print("Has session token:", frozen.token is not None)

Session profile: hcp
Credential method: shared-credentials-file
Access-key prefix: AKIA
Access-key length: 20
Access-key fingerprint: d7aebf79a361
Has session token: False


In [2]:
import configparser
import hashlib
from pathlib import Path

credentials_path = Path.home() / ".aws" / "credentials"

parser = configparser.ConfigParser()
parser.read(credentials_path)

file_access_key = (
    parser["hcp"]["aws_access_key_id"]
    .strip()
)

print("Credentials-file prefix:", file_access_key[:4])
print("Credentials-file length:", len(file_access_key))
print(
    "Credentials-file fingerprint:",
    hashlib.sha256(file_access_key.encode()).hexdigest()[:12],
)

Credentials-file prefix: AKIA
Credentials-file length: 20
Credentials-file fingerprint: d7aebf79a361


In [3]:
s3 = session.client(
    "s3",
    region_name="us-east-1",
)

response = s3.list_objects_v2(
    Bucket="hcp-openaccess",
    Prefix="HCP_1200/",
    Delimiter="/",
    MaxKeys=10,
)

print("Connected successfully.")
print(
    [
        item["Prefix"]
        for item in response.get("CommonPrefixes", [])
    ]
)

Connected successfully.
['HCP_1200/100206/', 'HCP_1200/100307/', 'HCP_1200/100408/', 'HCP_1200/100610/', 'HCP_1200/101006/', 'HCP_1200/101107/', 'HCP_1200/101309/', 'HCP_1200/101410/', 'HCP_1200/101915/']


In [4]:
SUBJECT_IDS = [
    "100307",
    "100408",
]

subject_id = SUBJECT_IDS[0]

motor_prefix = (
    f"HCP_1200/{subject_id}/"
    "MNINonLinear/Results/tfMRI_MOTOR_LR/"
)

print("Inspecting:", motor_prefix)

Inspecting: HCP_1200/100307/MNINonLinear/Results/tfMRI_MOTOR_LR/


In [5]:
import pandas as pd


def list_s3_objects(
    *,
    client,
    bucket: str,
    prefix: str,
) -> pd.DataFrame:
    """List all S3 objects under a prefix."""

    paginator = client.get_paginator("list_objects_v2")
    rows: list[dict] = []

    for page in paginator.paginate(
        Bucket=bucket,
        Prefix=prefix,
    ):
        for item in page.get("Contents", []):
            rows.append(
                {
                    "key": item["Key"],
                    "size_bytes": int(item["Size"]),
                    "size_mb": item["Size"] / (1024**2),
                    "last_modified": item["LastModified"],
                }
            )

    return pd.DataFrame(rows)


motor_objects = list_s3_objects(
    client=s3,
    bucket="hcp-openaccess",
    prefix=motor_prefix,
)

print("Number of objects:", len(motor_objects))

motor_objects[
    ["key", "size_mb"]
].sort_values("key")

Number of objects: 28


,key,size_mb
0,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.000006
1,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.000106
2,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.000024
3,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.000024
4,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.000024
5,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.000024
6,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.000024
7,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.042569
8,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.002411
9,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.000009


In [6]:
selected_motor_objects = motor_objects[
    motor_objects["key"].str.endswith(
        (
            "tfMRI_MOTOR_LR.nii.gz",
            "Movement_Regressors.txt",
            "Movement_Regressors_dt.txt",
        )
    )
    |
    motor_objects["key"].str.contains("/EVs/")
].copy()

selected_motor_objects = (
    selected_motor_objects
    .sort_values("key")
    .reset_index(drop=True)
)

selected_motor_objects[
    ["key", "size_mb"]
]

,key,size_mb
0,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.000006
1,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.000106
2,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.000024
3,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.000024
4,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.000024
5,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.000024
6,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.000024
7,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.036022
8,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,0.036022
9,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,224.333204


In [7]:
total_size_gb = (
    selected_motor_objects["size_bytes"].sum()
    / (1024**3)
)

print("Selected files:", len(selected_motor_objects))
print(f"Estimated download size: {total_size_gb:.2f} GB")

Selected files: 10
Estimated download size: 0.22 GB


In [8]:
from pathlib import Path
from typing import Iterable


PROJECT_ROOT = Path.cwd().resolve()

for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "environment.yml").exists():
        PROJECT_ROOT = candidate
        break

RAW_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "hcp_ya_s1200"
)

RAW_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def local_path_for_s3_key(
    key: str,
    *,
    dataset_prefix: str,
    destination_root: Path,
) -> Path:
    """Map an HCP S3 key to a local project path."""

    relative_path = Path(key).relative_to(
        dataset_prefix
    )

    return destination_root / relative_path


def download_s3_objects(
    *,
    client,
    bucket: str,
    keys: Iterable[str],
    dataset_prefix: str,
    destination_root: Path,
    overwrite: bool = False,
) -> pd.DataFrame:
    """Download selected S3 objects with skip-existing behavior."""

    records: list[dict] = []

    for key in keys:
        local_path = local_path_for_s3_key(
            key,
            dataset_prefix=dataset_prefix,
            destination_root=destination_root,
        )

        local_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        if local_path.exists() and not overwrite:
            status = "skipped_existing"
        else:
            print("Downloading:", key)

            client.download_file(
                bucket,
                key,
                str(local_path),
            )

            status = "downloaded"

        records.append(
            {
                "key": key,
                "local_path": str(local_path),
                "status": status,
                "exists": local_path.exists(),
                "size_bytes_local": (
                    local_path.stat().st_size
                    if local_path.exists()
                    else None
                ),
            }
        )

    return pd.DataFrame(records)

In [9]:
DOWNLOAD_FILES = True

In [10]:
if DOWNLOAD_FILES:
    download_results = download_s3_objects(
        client=s3,
        bucket="hcp-openaccess",
        keys=selected_motor_objects["key"],
        dataset_prefix="HCP_1200",
        destination_root=RAW_DATA_DIR,
    )

    display(download_results)
else:
    print(
        "Dry run only. Review the selected files and "
        "download size, then set DOWNLOAD_FILES = True."
    )

,key,local_path,status,exists,size_bytes_local
0,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,/Users/srinivasgovindasurampudi/Projects/neuro...,skipped_existing,True,6
1,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,/Users/srinivasgovindasurampudi/Projects/neuro...,skipped_existing,True,111
2,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,/Users/srinivasgovindasurampudi/Projects/neuro...,skipped_existing,True,25
3,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,/Users/srinivasgovindasurampudi/Projects/neuro...,skipped_existing,True,25
4,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,/Users/srinivasgovindasurampudi/Projects/neuro...,skipped_existing,True,25
5,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,/Users/srinivasgovindasurampudi/Projects/neuro...,skipped_existing,True,25
6,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,/Users/srinivasgovindasurampudi/Projects/neuro...,skipped_existing,True,25
7,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,/Users/srinivasgovindasurampudi/Projects/neuro...,skipped_existing,True,37772
8,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,/Users/srinivasgovindasurampudi/Projects/neuro...,skipped_existing,True,37772
9,HCP_1200/100307/MNINonLinear/Results/tfMRI_MOT...,/Users/srinivasgovindasurampudi/Projects/neuro...,skipped_existing,True,235230414
